In [1]:
import pandas as pd   # for loading and handling tabular data
import numpy as np    # for numerical operations
import json           # for saving small config values like the threshold
import joblib         # for saving Python objects like models and preprocessors

# split data into train/validation
from sklearn.model_selection import train_test_split

# preprocessing tools
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

# final chosen model
from xgboost import XGBClassifier

In [2]:
# Load the main transaction dataset
train_trans = pd.read_csv("../data/raw/train_transaction.csv")

# Load the identity dataset
train_id = pd.read_csv("../data/raw/train_identity.csv")

# Merge them on TransactionID so each transaction gets matching identity info
df = train_trans.merge(train_id, on="TransactionID", how="left")

# Check shape and preview
print(df.shape)
df.head()

(590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [23]:
# y is the target we want to predict
y = df["isFraud"]

# X contains the input features
# Drop the target column and the transaction ID column
X = df.drop(columns=["isFraud", "TransactionID"])

# Print shapes so we can verify
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (590540, 432)
y shape: (590540,)


In [24]:
# Compute the fraction of missing values in each feature column
missing_pct = X.isnull().mean()

# Find columns where more than 90% of values are missing
# We will remove these so the final pipeline matches what we used before
high_missing_cols = missing_pct[missing_pct > 0.90].index.tolist()

# Drop those columns
X = X.drop(columns=high_missing_cols)

# Print how many were dropped and the new shape
print("Dropped columns:", len(high_missing_cols))
print("New X shape:", X.shape)

Dropped columns: 12
New X shape: (590540, 420)


In [25]:
# Get a list of numeric feature columns
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()

# Get a list of categorical/text feature columns
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Print counts so we can verify
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 391
Categorical features: 29


C:\Users\hariv\AppData\Local\Temp\ipykernel_42008\2894317547.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [26]:
# Split the data into training and validation sets
# We are doing this the same way as before so the saved model setup matches
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Print shapes and fraud rates to verify
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Fraud rate train:", y_train.mean())
print("Fraud rate val:", y_val.mean())

X_train: (472432, 420)
X_val: (118108, 420)
Fraud rate train: 0.03498916246147594
Fraud rate val: 0.0349933958749619


In [27]:
# Create an imputer for numeric columns
# We use the median because it is robust to skewed values and outliers
num_imputer = SimpleImputer(strategy="median")

# Fit the imputer on the training numeric columns
num_imputer.fit(X_train[numeric_features])

# Transform the training and validation numeric data
X_train_num = pd.DataFrame(
    num_imputer.transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index
)

X_val_num = pd.DataFrame(
    num_imputer.transform(X_val[numeric_features]),
    columns=numeric_features,
    index=X_val.index
)

# Print shapes to verify
print("X_train_num shape:", X_train_num.shape)
print("X_val_num shape:", X_val_num.shape)

X_train_num shape: (472432, 391)
X_val_num shape: (118108, 391)


In [28]:
# Create an imputer for categorical columns
# Missing text values become the string "MISSING"
cat_imputer = SimpleImputer(strategy="constant", fill_value="MISSING")

# Fit the categorical imputer on the training categorical columns
cat_imputer.fit(X_train[categorical_features])

# Fill missing values in train and validation categorical data
X_train_cat_filled = pd.DataFrame(
    cat_imputer.transform(X_train[categorical_features]),
    columns=categorical_features,
    index=X_train.index
)

X_val_cat_filled = pd.DataFrame(
    cat_imputer.transform(X_val[categorical_features]),
    columns=categorical_features,
    index=X_val.index
)

# Create an ordinal encoder
# This turns category strings into integer codes
# unknown_value=-1 means unseen categories at prediction time will map to -1
cat_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

# Fit the encoder on training categorical data
cat_encoder.fit(X_train_cat_filled)

# Transform train and validation categorical data
X_train_cat = pd.DataFrame(
    cat_encoder.transform(X_train_cat_filled),
    columns=categorical_features,
    index=X_train.index
)

X_val_cat = pd.DataFrame(
    cat_encoder.transform(X_val_cat_filled),
    columns=categorical_features,
    index=X_val.index
)

# Print shapes to verify
print("X_train_cat shape:", X_train_cat.shape)
print("X_val_cat shape:", X_val_cat.shape)

X_train_cat shape: (472432, 29)
X_val_cat shape: (118108, 29)


In [29]:
# Combine the processed numeric columns and processed categorical columns
# This recreates the final model-ready feature tables
X_train_processed = pd.concat([X_train_num, X_train_cat], axis=1)
X_val_processed = pd.concat([X_val_num, X_val_cat], axis=1)

# Print shapes to verify
print("X_train_processed:", X_train_processed.shape)
print("X_val_processed:", X_val_processed.shape)

X_train_processed: (472432, 420)
X_val_processed: (118108, 420)


In [30]:
# Count how many non-fraud examples are in the training set
neg_count = (y_train == 0).sum()

# Count how many fraud examples are in the training set
pos_count = (y_train == 1).sum()

# Compute the class imbalance ratio
# This will help XGBoost pay more attention to the fraud class
scale_pos_weight = neg_count / pos_count

# Print values
print("Negative count:", neg_count)
print("Positive count:", pos_count)
print("scale_pos_weight:", scale_pos_weight)

Negative count: 455902
Positive count: 16530
scale_pos_weight: 27.580278281911674


In [31]:
# Create the final XGBoost model using the same settings that won earlier
xgb_model = XGBClassifier(
    n_estimators=300,             # number of boosting trees
    max_depth=6,                  # maximum depth of each tree
    learning_rate=0.1,            # contribution of each tree
    subsample=0.8,                # use 80% of rows per tree
    colsample_bytree=0.8,         # use 80% of columns per tree
    objective="binary:logistic",  # binary classification with probability output
    eval_metric="logloss",        # training evaluation metric
    random_state=42,              # keeps results reproducible
    scale_pos_weight=scale_pos_weight,  # handle class imbalance
    n_jobs=-1                     # use all available CPU cores
)

# Train the model on the processed training data
xgb_model.fit(X_train_processed, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [32]:
# Sanity check before saving artifacts
print("Numeric features count:", len(numeric_features))
print("Categorical features count:", len(categorical_features))
print("Processed feature count:", len(X_train_processed.columns))

print("First 5 numeric features:", numeric_features[:5])
print("First 5 categorical features:", categorical_features[:5])

Numeric features count: 391
Categorical features count: 29
Processed feature count: 420
First 5 numeric features: ['TransactionDT', 'TransactionAmt', 'card1', 'card2', 'card3']
First 5 categorical features: ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain']


In [34]:
# Save the trained XGBoost model
joblib.dump(xgb_model, "../models/xgb_model.pkl")

# Save the numeric imputer
joblib.dump(num_imputer, "../models/num_imputer.pkl")

# Save the categorical imputer
joblib.dump(cat_imputer, "../models/cat_imputer.pkl")

# Save the categorical encoder
joblib.dump(cat_encoder, "../models/cat_encoder.pkl")

# Save the numeric and categorical feature name lists
joblib.dump(numeric_features, "../models/numeric_features.pkl")
joblib.dump(categorical_features, "../models/categorical_features.pkl")

# Save the final processed column order
# This is important so the app can recreate the exact same feature layout
joblib.dump(X_train_processed.columns.tolist(), "../models/final_feature_order.pkl")

print("Model and preprocessing objects saved.")

Model and preprocessing objects saved.


In [35]:
# Store the final threshold and risk band cutoffs in a small config dictionary
model_config = {
    "final_threshold": 0.8,   # chosen from threshold tuning
    "low_risk_max": 0.30,     # below 0.30 = Low Risk
    "medium_risk_max": 0.70   # 0.30 to <0.70 = Medium Risk, 0.70+ = High Risk
}

# Save the config as a JSON file
with open("../models/model_config.json", "w") as f:
    json.dump(model_config, f, indent=4)

print("Threshold and risk band config saved.")

Threshold and risk band config saved.


In [36]:
# Load the saved model back from disk
loaded_model = joblib.load("../models/xgb_model.pkl")

# Load the saved preprocessors and config
loaded_num_imputer = joblib.load("../models/num_imputer.pkl")
loaded_cat_imputer = joblib.load("../models/cat_imputer.pkl")
loaded_cat_encoder = joblib.load("../models/cat_encoder.pkl")
loaded_numeric_features = joblib.load("../models/numeric_features.pkl")
loaded_categorical_features = joblib.load("../models/categorical_features.pkl")
loaded_feature_order = joblib.load("../models/final_feature_order.pkl")

with open("../models/model_config.json", "r") as f:
    loaded_config = json.load(f)

# Print quick checks so we know everything saved correctly
print(type(loaded_model))
print("Numeric features loaded:", len(loaded_numeric_features))
print("Categorical features loaded:", len(loaded_categorical_features))
print("Final feature order length:", len(loaded_feature_order))
print("Loaded config:", loaded_config)

<class 'xgboost.sklearn.XGBClassifier'>
Numeric features loaded: 391
Categorical features loaded: 29
Final feature order length: 420
Loaded config: {'final_threshold': 0.8, 'low_risk_max': 0.3, 'medium_risk_max': 0.7}
